# Screenplay Analysis — Sentiment, Genre & Viability Model Training



## 1. Local setup


In [24]:
import os
from pathlib import Path

REPO_ROOT = Path(r"D:\Griffith\Dissertation\Screenplay_Analysis")
OMDB_API_KEY = '1f29e936'

os.chdir(REPO_ROOT)
print("Working directory set to:", os.getcwd())


Working directory set to: D:\Griffith\Dissertation\Screenplay_Analysis


# Part A — Sentiment Model

Trains the RoBERTa-based scene-level sentiment model and generates a sentiment arc.

## 2. Install dependencies

`torchvision` is uninstalled first to avoid a version conflict with the `torch` build Colab ships by default.

In [ ]:
!pip uninstall -y torchvision
!pip install transformers datasets scikit-learn matplotlib pdfplumber


## 3. Train the sentiment model

Combines Rotten Tomatoes + IMDB by default.

In [ ]:
!python nlp_pipeline/train_sentiment_model.py

GPU found: Tesla T4

Loading Rotten Tomatoes dataset...
README.md: 100% 7.46k/7.46k [00:00<00:00, 22.2MB/s]

train.parquet: downloading bytes:   0% 0.00/699k [00:00<?, ?B/s]
train.parquet: downloading bytes: 100% 697k/697k [00:01<00:00, 459kB/s, 67.4kB/s  ] 
train.parquet: reconstructing file: 100% 699k/699k [00:01<00:00, 460kB/s, 67.6kB/s  ]

validation.parquet: downloading bytes:   0% 0.00/90.0k [00:00<?, ?B/s]
validation.parquet: downloading bytes: 100% 88.9k/88.9k [00:01<00:00, 65.4kB/s, 8.63kB/s  ]
validation.parquet: reconstructing file: 100% 90.0k/90.0k [00:01<00:00, 66.2kB/s, 8.74kB/s  ]

test.parquet: downloading bytes:   0% 0.00/92.2k [00:00<?, ?B/s]
test.parquet: downloading bytes: 100% 91.4k/91.4k [00:01<00:00, 69.2kB/s, 8.88kB/s  ]
test.parquet: reconstructing file: 100% 92.2k/92.2k [00:01<00:00, 69.8kB/s, 8.96kB/s  ]
Generating train split: 100% 8530/8530 [00:00<00:00, 175721.44 examples/s]
Generating validation split: 100% 1066/1066 [00:00<00:00, 285622.08 examples/s]
Ge

## 4. Example: Generate and plot a sentiment arc


In [ ]:
!python nlp_pipeline/sentiment_arc.py data/Titanic.txt
!python nlp_pipeline/plot_sentiment_arc.py Titanic_sentiment_arc.json


Loading fine-tuned sentiment model from /content/Screenplay_Analysis_and_Script_Coverage_Tool/models/roberta-sentiment-finetuned ...
Loading weights: 100% 201/201 [00:00<00:00, 4619.76it/s]

Scoring 194 scenes for 'Titanic' (model: fine-tuned-roberta)...
  Scored 20/194 scenes...
  Scored 40/194 scenes...
  Scored 60/194 scenes...
  Scored 80/194 scenes...
  Scored 100/194 scenes...
  Scored 120/194 scenes...
  Scored 140/194 scenes...
  Scored 160/194 scenes...
  Scored 180/194 scenes...
  Done. All 194 scenes scored.

  SENTIMENT ARC — Titanic  (model: fine-tuned-roberta)
  Scenes analysed   : 194
  Average sentiment : +0.0865
  Positive scenes   : 106
  Negative scenes   : 88
  Turning points    : 34

  Most positive scene:
    24 INT. LAB DECK, PRESERVATION AREA (+0.9978)
  Most negative scene (darkest moment):
    164 EXT. BOAT DECK (-0.9932)

  Emotional arc (every 10th scene):
    Scene   0: [-████████            ] -0.414
    Scene   9: [-████                ] -0.209
    Scene  

# Part B — Genre Classifier

Trains and evaluates the multi-label genre classifier using TF-IDF text features and structural features.

# Genre Classifier Training

Trains the multi-label genre classifier: TF-IDF text features + structural features, one XGBoost model per genre, with per-genre `scale_pos_weight` to correct class imbalance.


## 1. Install dependencies


In [4]:
!pip install --upgrade pip
!pip install scikit-learn xgboost joblib scipy

  Using cached pip-26.2.1-py3-none-any.whl.metadata (4.6 kB)
Using cached pip-26.2.1-py3-none-any.whl (1.8 MB)



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: To modify pip, please run the following command:
D:\Griffith\Dissertation\Screenplay_Analysis\venv\Scripts\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Prepare the labeled corpus



In [5]:
!python nlp_pipeline/fetch_genre_labels.py --join dataset/corpus_clean_characters.jsonl


Fetching genre labels for 18 genres from IMSDB...

  Fetching Action... 351 titles
  Fetching Adventure... 225 titles
  Fetching Animation... 57 titles
  Fetching Comedy... 431 titles
  Fetching Crime... 242 titles
  Fetching Drama... 709 titles
  Fetching Family... 68 titles
  Fetching Fantasy... 148 titles
  Fetching Film-Noir... 9 titles
  Fetching Horror... 171 titles
  Fetching Musical... 36 titles
  Fetching Mystery... 135 titles
  Fetching Romance... 228 titles
  Fetching Sci-Fi... 193 titles
  Fetching Short... 8 titles
  Fetching Thriller... 425 titles
  Fetching War... 41 titles
  Fetching Western... 23 titles

Saved 1300 titles' genre labels to D:\Griffith\Dissertation\Screenplay_Analysis\dataset\genre_labels.json
(1072 of 1300 titles have more than one genre)

Joined onto corpus: 1117 matched, 0 unmatched (no genre found)
Written to D:\Griffith\Dissertation\Screenplay_Analysis\dataset\corpus_with_genres.jsonl


## 3. Train the genre classifier

Runs 5-fold cross-validation, applies per-genre `scale_pos_weight` to correct class imbalance, and saves the trained model plus a full metrics report.


In [6]:
!python nlp_pipeline/train_genre_classifier.py dataset/corpus_with_genres.jsonl


Screenplays with usable genre labels: 1116
Genres kept (>= 5 examples): ['Action', 'Adventure', 'Animation', 'Comedy', 'Crime', 'Drama', 'Family', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']

Running 5-fold cross-validation...
  Fold 1: macro F1 = 0.391
  Fold 2: macro F1 = 0.442
  Fold 3: macro F1 = 0.418
  Fold 4: macro F1 = 0.462
  Fold 5: macro F1 = 0.414

  Mean macro F1 across 5 folds: 0.425 (+/- 0.025)

  Per-genre performance:
    Action       precision=0.80  recall=0.77  f1=0.78  support=304
    Adventure    precision=0.78  recall=0.54  f1=0.64  support=184
    Animation    precision=0.75  recall=0.30  f1=0.43  support=40
    Comedy       precision=0.71  recall=0.58  f1=0.64  support=358
    Crime        precision=0.67  recall=0.47  f1=0.56  support=218
    Drama        precision=0.75  recall=0.80  f1=0.77  support=629
    Family       precision=0.25  recall=0.05  f1=0.09  support=39
    Fantasy      precision=0.49

## 5. Inspect the results

Loads the saved report directly.

In [7]:
import json

with open('dataset/genre_classifier_report.json') as f:
    report = json.load(f)

print('Screenplays used:', report['screenplay_count'])
print('Mean macro F1:', round(report['mean_macro_f1'], 3))
print()
print(f"{'Genre':<12} {'Precision':>10} {'Recall':>10} {'F1':>8} {'Support':>8}")
for genre, m in report['per_genre_metrics'].items():
    print(f"{genre:<12} {m['precision']:>10.3f} {m['recall']:>10.3f} "
          f"{m['f1-score']:>8.3f} {int(m['support']):>8d}")


Screenplays used: 1116
Mean macro F1: 0.425

Genre         Precision     Recall       F1  Support
Action            0.799      0.770    0.784      304
Adventure         0.780      0.538    0.637      184
Animation         0.750      0.300    0.429       40
Comedy            0.710      0.575    0.636      358
Crime             0.673      0.472    0.555      218
Drama             0.749      0.798    0.773      629
Family            0.250      0.051    0.085       39
Fantasy           0.486      0.159    0.240      113
Film-Noir         0.000      0.000    0.000        8
Horror            0.762      0.526    0.623      152
Musical           0.250      0.125    0.167       24
Mystery           0.778      0.064    0.118      110
Romance           0.582      0.328    0.420      195
Sci-Fi            0.778      0.497    0.606      169
Thriller          0.700      0.653    0.676      375
War               0.462      0.171    0.250       35
Western           0.714      0.250    0.370       20


# Part C — Viability Regressor
Trains the XGBoost regression model that predicts IMDb rating from screenplay features.

## 1. Fetch real IMDb ratings via OMDb

Requires a free OMDb API key (https://www.omdbapi.com/apikey.aspx).


In [25]:
if not OMDB_API_KEY:
    raise ValueError(
        "OMDB_API_KEY is empty."
    )

!python nlp_pipeline/fetch_viability_labels.py --api-key {OMDB_API_KEY} dataset/corpus_with_genres.jsonl --join


Reading corpus from: D:\Griffith\Dissertation\Screenplay_Analysis\dataset\corpus_with_genres.jsonl
1117 total scripts to look up.
Only doing 1000 this run (free-tier daily cap). Re-run to continue with the remaining titles.
  [1/1000] 10 Things I Hate About You               rating=7.4
  [2/1000] 12 and Holding                           rating=7.4
  [3/1000] 12 Monkeys                               rating=8.0
  [4/1000] 12 Years a Slave                         rating=8.1
  [5/1000] 12                                       rating=7.5
  [6/1000] 127_Hours                                rating=7.5
  [7/1000] 1492_ Conquest of Paradise               rating=6.4
  [8/1000] 15 Minutes                               rating=6.1
  [9/1000] 17 Again                                 rating=6.4
  [10/1000] 187                                      rating=None
  [11/1000] 2001_ A Space Odyssey                    rating=8.3
  [12/1000] 2012                                     rating=5.9
  [13/1000] 20th

## 2. Train the viability regressor

Runs an automatic A/B comparison: text+structure only, versus text+structure+genre, against a naive baseline (always predict the corpus mean rating), so the effect of adding genre as a feature is a direct, honest before/after rather than a single unverified number.


In [26]:
!python nlp_pipeline/train_viability_model.py dataset/corpus_with_viability.jsonl


Screenplays with a usable IMDb rating: 841 (276 dropped -- no OMDb match or no rating)

Naive baseline MAE (always guess the average): 0.750

Running 5-fold cross-validation -- text + structure only...
  Fold 1: MAE=0.714  RMSE=0.919  R²=0.038
  Fold 2: MAE=0.753  RMSE=0.954  R²=-0.014
  Fold 3: MAE=0.716  RMSE=0.932  R²=0.018
  Fold 4: MAE=0.793  RMSE=1.048  R²=0.023
  Fold 5: MAE=0.706  RMSE=0.896  R²=0.013

Running 5-fold cross-validation -- text + structure + genre...
  Fold 1: MAE=0.675  RMSE=0.867  R²=0.142
  Fold 2: MAE=0.762  RMSE=0.954  R²=-0.014
  Fold 3: MAE=0.687  RMSE=0.907  R²=0.069
  Fold 4: MAE=0.779  RMSE=1.023  R²=0.068
  Fold 5: MAE=0.711  RMSE=0.915  R²=-0.030

  Naive baseline MAE:                0.750
  Text+structure MAE:                0.736  (R²=0.015, 1.8% over baseline)
  Text+structure+genre MAE:          0.723  (R²=0.047, 3.6% over baseline)
  Effect of adding genre:            +0.0134 MAE (helps)

  10 Things I Hate About You: actual=7.4, predicted=7.26

 

## 3. Inspect the results

This is the real evidence of training: MAE/R² for both model versions, compared against the naive baseline.


In [15]:
import json

with open('dataset/viability_report.json') as f:
    report = json.load(f)

print('Screenplays with a usable rating:', report['screenplay_count'])
print('Dropped (no OMDb match):', report['dropped_no_rating'])
print('Naive baseline MAE:', round(report['naive_baseline_mae'], 3))
print()
base = report['text_and_structure_only']
print('Text + structure       -- MAE:', round(base['mean_mae'], 3),
      ' R2:', round(base['mean_r2'], 3))
if report.get('text_structure_and_genre'):
    genre = report['text_structure_and_genre']
    print('Text + structure + genre -- MAE:', round(genre['mean_mae'], 3),
          ' R2:', round(genre['mean_r2'], 3))


Screenplays with a usable rating: 841
Dropped (no OMDb match): 276
Naive baseline MAE: 0.75

Text + structure       -- MAE: 0.736  R2: 0.015
Text + structure + genre -- MAE: 0.723  R2: 0.047
